In [20]:
import numpy as np
from flight_optimizer import *
import itertools

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
def find_min_delta(solution: FlightSolution):
    min_gap = float('inf')
    for aircraft, flights in solution.assignment.items():
        for f1, f2 in itertools.combinations(flights, 2):
            if f2.arrival_time < f1.arrival_time:
                f1, f2 = f2, f1

            min_gap = min(min_gap, f2.departure_time - f1.arrival_time)

    return min_gap

In [23]:
def problem_path(d: float, p: int, h: int, test_index: int,max_d: int):
    return f"./data/With maintenance constraints/d={max_d}/DataCplex_density={d}_p={p}_h={h}_test_{test_index}.dat"


def solution_path(d: float, p: int, h: int, test_index: int, max_d: int):
    if max_d == 5:
        return f"./data/With maintenance constraints/d={max_d}/Optimal_Solution_density={d}_p={p}_h={h}_test_{test_index}_dmax_{max_d}.txt"
    else:
        return f"./data/With maintenance constraints/d={max_d}/Optimal_Solution_density={d}_p={p}_h={h}_test_{test_index}.txt"

In [24]:
# D = [0.5, 0.7, 1]
# P = [10, 20, 30, 40]
# H = [7, 15, 21, 30]

# print('d\t| p\t| h\t| i\t| min_gap')
# for density, planes, horizon, index in itertools.product(D, P, H, range(10)):
#     try:
#         problem = FlightProblem.from_file(problem_path(density, planes, horizon, index))
#         solution = FlightSolution.from_file(solution_path(density, planes, horizon, index), problem)
#         min_gap = find_min_delta(solution)
#         print(f'{density}\t| {planes}\t| {horizon}\t| {index}\t| {min_gap}')
#     except Exception as e:
#         print(f'{density}\t| {planes}\t| {horizon}\t| {index}\t| {type(e)}')


In [25]:
instance = (1,10,15,0, 4)

In [26]:
instances = [
    (0.5, 10, 30, 8),
    (0.5, 20, 21, 1),
    (0.5, 20, 30, 3),
    (0.5, 20, 30, 5),
    (0.5, 30, 7, 1),
    (0.5, 30, 7, 5),
    (0.5, 30, 7, 7),
    (0.5, 30, 7, 8),
    (0.5, 30, 15, 2),
]

In [27]:
reporter = Reporter()

In [29]:
problem = FlightProblemMaintenance.from_file(problem_path(*instance), instance[4])
for flight in problem.flights:
    flight.arrival_time += 30

In [30]:
reduced_solver = ReducedFlowSolver()
solution_found_1 = reduced_solver.solve_maintenance(problem)

number of decision variables : 5730
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 4513 rows, 5730 columns and 187130 nonzeros
Model fingerprint: 0x51a47abb
Variable types: 0 continuous, 5730 integer (5730 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+03, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Presolve removed 1131 rows and 593 columns
Presolve time: 0.33s
Presolved: 3382 rows, 5137 columns, 112577 nonzeros
Variable types: 0 continuous, 5137 integer (5130 binary)

Root relaxation: objective 3.540797e+06, 13846 iterations, 1.37 seconds (2.25 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd  

In [31]:
solution = FlightSolutionMaintenance.from_file(solution_path(*instance), problem)

In [33]:
(solution_found_1.cost - solution.cost)

np.float64(-6191.0)

In [34]:
solution_found_1.maintenances

{0: [(3, Airport(id=3, name='MAD')),
  (6, Airport(id=3, name='MAD')),
  (10, Airport(id=6, name='GLA'))],
 1: [(3, Airport(id=3, name='MAD')),
  (7, Airport(id=3, name='MAD')),
  (11, Airport(id=3, name='MAD'))],
 2: [(3, Airport(id=16, name='MUC')),
  (7, Airport(id=16, name='MUC')),
  (11, Airport(id=16, name='MUC'))],
 3: [(7, Airport(id=3, name='MAD')),
  (11, Airport(id=3, name='MAD')),
  (3, Airport(id=16, name='MUC'))],
 4: [(2, Airport(id=3, name='MAD')),
  (6, Airport(id=3, name='MAD')),
  (10, Airport(id=3, name='MAD'))],
 5: [(2, Airport(id=3, name='MAD')),
  (6, Airport(id=3, name='MAD')),
  (10, Airport(id=6, name='GLA'))],
 6: [(2, Airport(id=3, name='MAD')),
  (10, Airport(id=3, name='MAD')),
  (6, Airport(id=6, name='GLA'))],
 7: [(2, Airport(id=3, name='MAD')),
  (6, Airport(id=3, name='MAD')),
  (10, Airport(id=3, name='MAD'))],
 8: [(3, Airport(id=3, name='MAD')),
  (7, Airport(id=3, name='MAD')),
  (11, Airport(id=3, name='MAD'))],
 9: [(3, Airport(id=3, name='MAD'